# 🎹 Aria — Chopin Style Finetuning
**Goal:** Finetune EleutherAI Aria on Chopin MIDI files so it generates music with Chopin's specific harmonic language, ornamental style, and emotional depth.

**Method:** LoRA (Low-Rank Adaptation) — efficient finetuning that modifies a small number of parameters, preserving the model's piano/musical knowledge while shifting its style toward Chopin.

### What you need
- A folder of **Chopin MIDI files** (80–200 files is enough). Performance MIDIs are better than score MIDIs.
- **Colab A100 GPU** (recommended) or T4 (slower but works)
- This will take **1–3 hours** for a good finetune on T4

### Where to get Chopin MIDIs (free)
- [Kunstderfuge](http://www.kunstderfuge.com/chopin.htm) — large collection of performance MIDIs
- [Mutopia Project](https://www.mutopiaproject.org/cgibin/make-table.cgi?Composer=ChopinFF) — free, clean scores
- [Piano MIDI de](http://www.piano-midi.de/chopin.htm) — high quality performance MIDIs
- [IMSLP](https://imslp.org/) — search "Chopin" → MIDI files section

**Tip:** Mix different genres (nocturnes, mazurkas, waltzes, etudes, preludes) for a more general Chopin style rather than one that only sounds like one piece type.

---

**Runtime:** Use **A100** (Colab Pro) if available. T4 will work but is ~3x slower.

## Step 1 — Install Dependencies

In [1]:
!pip install -q git+https://github.com/EleutherAI/aria-utils.git
!pip install -q transformers safetensors huggingface_hub peft accelerate
!pip install -q pretty_midi tqdm
!pip install -q "torchao>=0.16.0"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ Dependencies installed")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 52.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.5 MB/s eta 0:00:00

✅ Dependencies installed
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## Step 2 — Upload Your Chopin MIDI Dataset

Upload a zip file containing all your Chopin MIDI files, or upload them individually.

In [2]:
import os, zipfile, glob
from google.colab import files

MIDI_DIR = "chopin_midi"
os.makedirs(MIDI_DIR, exist_ok=True)

print("Upload your Chopin MIDI files (zip or individual .mid files):")
print("→ If you have a zip, upload that. If individual files, upload them all at once.")
print()

uploaded = files.upload()

for fname, data in uploaded.items():
    fpath = os.path.join(MIDI_DIR, fname)
    with open(fpath, 'wb') as f:
        f.write(data)

    # Unzip if it's a zip
    if fname.endswith('.zip'):
        print(f"Extracting {fname}...")
        with zipfile.ZipFile(fpath, 'r') as zf:
            zf.extractall(MIDI_DIR)
        os.remove(fpath)

# Collect all MIDI files recursively
midi_files = glob.glob(f"{MIDI_DIR}/**/*.mid", recursive=True) + \
             glob.glob(f"{MIDI_DIR}/**/*.midi", recursive=True) + \
             glob.glob(f"{MIDI_DIR}/*.mid") + \
             glob.glob(f"{MIDI_DIR}/*.midi")
midi_files = sorted(set(midi_files))

print(f"\n✅ Found {len(midi_files)} MIDI files")
if len(midi_files) < 20:
    print("⚠️  Less than 20 files — the model can still finetune but style transfer will be limited.")
    print("   Recommended: 80+ files for a recognizable Chopin style.")
elif len(midi_files) >= 80:
    print("👍 Great dataset size! This should produce a strong Chopin style.")

Upload your Chopin MIDI files (zip or individual .mid files):
→ If you have a zip, upload that. If individual files, upload them all at once.



Saving Chopin-PianoOnly+ShiftsAgumentaion.zip to Chopin-PianoOnly+ShiftsAgumentaion.zip
Extracting Chopin-PianoOnly+ShiftsAgumentaion.zip...

✅ Found 940 MIDI files
👍 Great dataset size! This should produce a strong Chopin style.


## Step 3 — Load the Base Model + Tokenizer

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import transformers.tokenization_utils as _tok_utils

print("Loading Aria model (downloading ~2.8 GB on first run)...")

# ── Fix 1: patch missing initializer_range ──
config = AutoConfig.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)
if not hasattr(config, "initializer_range"):
    config.initializer_range = 0.02

model = AutoModelForCausalLM.from_pretrained(
    "loubb/aria-medium-base",
    config=config,
    trust_remote_code=True,
    dtype=torch.float16,
)

# ── Load gen checkpoint as starting point ──
print("Loading generation checkpoint as starting point...")
gen_ckpt_path = hf_hub_download(
    repo_id="loubb/aria-medium-base",
    filename="model-gen.safetensors"
)
gen_state_dict = load_file(gen_ckpt_path)
model.load_state_dict(gen_state_dict, strict=False)

# ── Fix 2: inject BatchEncoding into tokenization_utils namespace ──
if not hasattr(_tok_utils, "BatchEncoding"):
    from transformers.tokenization_utils_base import BatchEncoding
    _tok_utils.BatchEncoding = BatchEncoding
    print("Patched BatchEncoding into tokenization_utils namespace")

tokenizer = AutoTokenizer.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)

# ── Fix 3: patch unimplemented save_vocabulary ──
tokenizer.save_vocabulary = lambda save_directory, filename_prefix=None: ()

print("✅ Base model loaded")

Loading Aria model (downloading ~2.8 GB on first run)...


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

configuration_aria.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/loubb/aria-medium-base:
- configuration_aria.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_aria.py: 0.00B [00:00, ?B/s]

...

Patched BatchEncoding into tokenization_utils namespace


tokenizer_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenization_aria.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/loubb/aria-medium-base:
- tokenization_aria.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


✅ Base model loaded


## Step 4 — Apply LoRA for Efficient Finetuning

In [5]:
from peft import LoraConfig, get_peft_model, TaskType

LORA_R       = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.1

# Aria's actual layer names (confirmed from inspection)
TARGET_MODULES = [
    "att_proj_linear",  # combined QKV + output attention projection
    "ff_gate_proj",     # feedforward gate
    "ff_up_proj",       # feedforward up
    "ff_down_proj",     # feedforward down
]

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model = model.to(DEVICE)
print("\n✅ LoRA applied — model is ready to finetune")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 26,738,688 || all params: 685,277,184 || trainable%: 3.9019

✅ LoRA applied — model is ready to finetune


## Step 5 — Prepare the Dataset

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

SEQ_LEN = 2048
STRIDE  = 1024

class ChopinMIDIDataset(Dataset):
    def __init__(self, midi_files, tokenizer, seq_len, stride):
        self.samples = []
        failed = 0

        print(f"Tokenizing {len(midi_files)} MIDI files...")
        for path in tqdm(midi_files):
            try:
                result = tokenizer.encode_from_file(path)

                # Aria tokenizer returns a BatchEncoding — extract input_ids
                if hasattr(result, '__getitem__') and 'input_ids' in result:
                    token_ids = result['input_ids']
                    # input_ids is a list of lists (batch dim) — unwrap it
                    if isinstance(token_ids, list) and isinstance(token_ids[0], list):
                        token_ids = token_ids[0]
                elif isinstance(result, list):
                    token_ids = result
                else:
                    token_ids = result.tolist()

                if len(token_ids) < 64:
                    failed += 1
                    continue

                # Chop into overlapping windows of SEQ_LEN
                for start in range(0, max(1, len(token_ids) - seq_len), stride):
                    chunk = token_ids[start : start + seq_len]
                    if len(chunk) < 64:
                        continue
                    if len(chunk) < seq_len:
                        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id else 0
                        chunk = chunk + [pad_id] * (seq_len - len(chunk))
                    self.samples.append(torch.tensor(chunk, dtype=torch.long))

            except Exception as e:
                failed += 1

        random.shuffle(self.samples)
        print(f"\n✅ Dataset ready: {len(self.samples)} training sequences")
        print(f"   From {len(midi_files) - failed} valid files ({failed} failed to parse)")
        print(f"   Approx. total music: ~{len(self.samples) * seq_len * 0.03 / 60:.0f} min")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return {"input_ids": ids, "labels": ids.clone()}


dataset = ChopinMIDIDataset(midi_files, tokenizer, SEQ_LEN, STRIDE)

if len(dataset) == 0:
    raise ValueError("No training samples were created! Check that your MIDI files are valid.")

Tokenizing 940 MIDI files...


100%|██████████| 940/940 [06:09<00:00,  2.54it/s]


✅ Dataset ready: 2750 training sequences
   From 940 valid files (0 failed to parse)
   Approx. total music: ~4224 min


## Step 6 — Train! 🎼

Adjust training settings below. For a first run, the defaults are safe.

In [8]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch

# ╔══════════════════════════════════════════╗
# ║        TRAINING CONFIGURATION           ║
# ╚══════════════════════════════════════════╝

NUM_EPOCHS       = 4
BATCH_SIZE       = 4
GRAD_ACCUM       = 4
LEARNING_RATE    = 2e-5
SAVE_STEPS       = 10
OUTPUT_DIR       = "aria_chopin_lora"

# ══════════════════════════════════════════

# Split dataset — 95% train, 5% validation
val_size        = int(len(dataset) * 0.05)
train_size      = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"Train sequences : {train_size}")
print(f"Val sequences   : {val_size}")

# ══════════════════════════════════════════

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,
    optim="adamw_torch_fused",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=200,
    eval_strategy="steps",   # evaluate every eval_steps
    eval_steps=10,                 # same frequency as save so best is never missed
    load_best_model_at_end=True,   # auto-keeps the checkpoint with lowest val loss
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    torch_compile=True,
)

# Data collator — pads/batches sequences for causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

class AriaTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop("num_items_in_batch", None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

trainer = AriaTrainer(        # ← was Trainer
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

# Estimate training time
steps_per_epoch = train_size // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * NUM_EPOCHS
print(f"\nTraining plan:")
print(f"  Dataset size    : {train_size} sequences")
print(f"  Steps per epoch : ~{steps_per_epoch}")
print(f"  Total steps     : ~{total_steps}")
print(f"  Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"\n  ⏱ Estimated time on A100: ~{total_steps * 1.5 / 60:.0f}–{total_steps * 2.5 / 60:.0f} min")
print()
print("Starting training...")
print("Watch the loss — it should drop from ~5–7 down to ~2–3 for a good finetune.")
print("Best checkpoint is saved automatically — you won't lose it this time.")
print()

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train sequences : 2613
Val sequences   : 137

Training plan:
  Dataset size    : 2613 sequences
  Steps per epoch : ~163
  Total steps     : ~815
  Effective batch : 16

  ⏱ Estimated time on A100: ~20–34 min

Starting training...
Watch the loss — it should drop from ~5–7 down to ~2–3 for a good finetune.
Best checkpoint is saved automatically — you won't lose it this time.



/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:2156: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(


Step,Training Loss,Validation Loss
10,3.790426,1.043524
20,3.645747,1.009362
30,3.910312,0.984706
40,3.511105,0.969840
50,3.406794,0.959119
60,3.547323,0.949146
70,3.721626,0.940672
80,3.223924,0.933655
90,3.647382,0.926955
100,3.849728,0.920583


TrainOutput(global_step=820, training_loss=2.9971723835642745, metrics={'train_runtime': 2400.4279, 'train_samples_per_second': 5.443, 'train_steps_per_second': 0.342, 'total_flos': 1.5846734701264896e+17, 'train_loss': 2.9971723835642745, 'epoch': 5.0})

## Step 7 — Save the LoRA Weights

In [15]:
from peft import PeftModel
import os

CHECKPOINTS_TO_EXPORT = [400, 550, 700, 790]
BASE_DIR = "/content/aria_chopin_lora"

for step in CHECKPOINTS_TO_EXPORT:
    ckpt_path = os.path.join(BASE_DIR, f"checkpoint-{step}")

    if not os.path.exists(ckpt_path):
        print(f"⚠️  checkpoint-{step} not found, skipping")
        continue

    # Load the LoRA weights from this checkpoint
    export_model = PeftModel.from_pretrained(model.base_model.model, ckpt_path)

    # Save clean adapter-only weights
    export_dir = f"/content/chopin_lora_step{step}"
    export_model.save_pretrained(export_dir)
    tokenizer.save_pretrained(export_dir)

    print(f"✅ Exported clean LoRA for step {step} → {export_dir}")

print("\nAll exported!")

✅ Exported clean LoRA for step 400 → /content/chopin_lora_step400
✅ Exported clean LoRA for step 550 → /content/chopin_lora_step550
✅ Exported clean LoRA for step 700 → /content/chopin_lora_step700
✅ Exported clean LoRA for step 790 → /content/chopin_lora_step790

All exported!


In [ ]:
160, 320, 500, 650, 770]

In [12]:
import os, shutil
from google.colab import files

CHECKPOINTS_TO_DOWNLOAD = [400, 550, 700, 790]

for step in CHECKPOINTS_TO_DOWNLOAD:
    export_dir = f"/content/chopin_lora_step{step}"

    if not os.path.exists(export_dir):
        print(f"⚠️  chopin_lora_step{step} not found, skipping")
        continue

    zip_path = f"/content/chopin_lora_step{step}"
    shutil.make_archive(zip_path, "zip", export_dir)
    print(f"📦 Zipping step {step} ({os.path.getsize(zip_path + '.zip')/1e6:.1f} MB)")

    files.download(zip_path + ".zip")
    print(f"✅ Done step {step}\n")

print("All done!")

📦 Zipping step 770 (98.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done step 770

All done!


In [16]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# Loss values for each checkpoint
CHECKPOINT_INFO = {
    400: {"val": 0.796423, "train": 2.934739},
    550: {"val": 0.768336, "train": 2.697661},
    700: {"val": 0.756792, "train": 2.610403},
    790: {"val": 0.755432, "train": 2.829171},
}

DRIVE_DIR = "/content/drive/MyDrive/chopin_lora_checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)

for step, losses in CHECKPOINT_INFO.items():
    export_dir = f"/content/chopin_lora_step{step}"

    if not os.path.exists(export_dir):
        print(f"⚠️  chopin_lora_step{step} not found, skipping")
        continue

    # e.g. chopin_lora_step160_val0.888_train3.141
    zip_name = f"chopin_lora_step{step}_val{losses['val']:.3f}_train{losses['train']:.3f}"
    zip_path = os.path.join(DRIVE_DIR, zip_name)
    shutil.make_archive(zip_path, "zip", export_dir)
    size = os.path.getsize(zip_path + ".zip") / 1e6
    print(f"✅ step {step} → {zip_name}.zip ({size:.1f} MB)")

print("\nAll done! Find them in MyDrive/chopin_lora_checkpoints")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ step 400 → chopin_lora_step400_val0.796_train2.935.zip (98.9 MB)
✅ step 550 → chopin_lora_step550_val0.768_train2.698.zip (98.9 MB)
✅ step 700 → chopin_lora_step700_val0.757_train2.610.zip (98.9 MB)
✅ step 790 → chopin_lora_step790_val0.755_train2.829.zip (98.9 MB)

All done! Find them in MyDrive/chopin_lora_checkpoints


In [13]:
import os

path = "/content/chopin_lora_step770.zip"

size_bytes = os.path.getsize(path)
size_mb = size_bytes / (1024**2)
size_gb = size_bytes / (1024**3)

print(f"Size: {size_bytes:,} bytes")
print(f"Size: {size_mb:.2f} MB")
print(f"Size: {size_gb:.2f} GB")

Size: 98,899,601 bytes
Size: 94.32 MB
Size: 0.09 GB


In [ ]:
LORA_SAVE_DIR = "aria_chopin_lora_final"
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)

print(f"✅ LoRA weights saved to '{LORA_SAVE_DIR}/'")
print()
print("Files saved:")
for f in os.listdir(LORA_SAVE_DIR):
    size = os.path.getsize(os.path.join(LORA_SAVE_DIR, f)) / 1e6
    print(f"  {f} ({size:.1f} MB)")
print()
print("💡 These are just the LoRA ADAPTER weights — much smaller than the full model!")
print("   To use them, you load the base model first, then apply these weights on top.")